# Conferência da legenda v7 contra a base

Este notebook verifica nomes e posições das colunas, tipos, domínios, faixas, preenchimento e estrutura da base. As contagens são agregadas. Identificadores são usados apenas internamente para testar unicidade e nunca têm seus valores exibidos ou contados por categoria. Categorias com frequência abaixo de 10 são agrupadas na saída para reduzir risco de identificação.

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
from IPython.display import display
from pandas.api.types import is_datetime64_any_dtype, is_integer_dtype, is_numeric_dtype

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
caminho_csv = os.getenv('BASE_CSV_PATH')
if not caminho_csv:
    raise EnvironmentError('Defina BASE_CSV_PATH para Base_academica.csv antes de executar este notebook.')
caminho_csv = Path(caminho_csv)
if not caminho_csv.is_file():
    raise FileNotFoundError('BASE_CSV_PATH não aponta para um arquivo existente.')

colunas_csv = pd.read_csv(caminho_csv, sep=';', encoding='utf-8-sig', nrows=0).columns.tolist()
identificadores_csv = [c for c in colunas_csv if c.startswith('HANDLE')]
df = pd.read_csv(caminho_csv, sep=';', encoding='utf-8-sig', decimal=',', low_memory=False, dtype={c: 'Int64' for c in identificadores_csv})
datas_csv = [c for c in colunas_csv if c.startswith('DATA')]
datas_invalidas = False
for coluna in datas_csv:
    original = df[coluna]
    convertida = pd.to_datetime(original, format='%d/%m/%y', errors='coerce')
    datas_invalidas = datas_invalidas or bool((original.notna() & convertida.isna()).any())
    df[coluna] = convertida
legenda = (RAIZ / 'data' / 'Legenda.md').read_text(encoding='utf-8-sig')
print(f'Base carregada: {len(df):,} linhas e {len(colunas_csv)} colunas no CSV.')
print(f'Todas as datas preenchidas seguem dd/mm/aa: {not datas_invalidas}')

Base carregada: 59,059 linhas e 66 colunas no CSV.
Todas as datas preenchidas seguem dd/mm/aa: True


## Nomes, posições e origem das linhas

A coluna `#` da legenda fornece a posição esperada. O comparativo mostra apenas metadados e nomes de colunas, nunca valores da base.

In [2]:
pares_legenda = []
for linha in legenda.splitlines():
    pares_legenda.extend((int(pos), nome) for pos, nome in re.findall(r'\|\s*(\d+)\s*\|\s*`([^`]+)`', linha))

posicoes_duplicadas = sorted({pos for pos, _ in pares_legenda if sum(p == pos for p, _ in pares_legenda) > 1})
nomes_duplicados = sorted({nome for _, nome in pares_legenda if sum(n == nome for _, n in pares_legenda) > 1})
legenda_ordenada = [nome for _, nome in sorted(pares_legenda)]
colunas_fora_da_legenda = [c for c in colunas_csv if c not in {nome for _, nome in pares_legenda}]
colunas_ausentes_no_csv = [nome for _, nome in pares_legenda if nome not in colunas_csv]

resumo_estrutura = pd.DataFrame([
    ('Colunas no CSV', len(colunas_csv)),
    ('Colunas numeradas na legenda', len(pares_legenda)),
    ('Nomes e posições conferem', colunas_csv == legenda_ordenada and not posicoes_duplicadas and not nomes_duplicados),
    ('Colunas extras no CSV', len(colunas_fora_da_legenda)),
    ('Colunas da legenda ausentes no CSV', len(colunas_ausentes_no_csv)),
] , columns=['verificação', 'resultado'])
display(resumo_estrutura)
if colunas_fora_da_legenda or colunas_ausentes_no_csv or posicoes_duplicadas or nomes_duplicados:
    print('Colunas extras no CSV:', colunas_fora_da_legenda)
    print('Colunas da legenda ausentes no CSV:', colunas_ausentes_no_csv)
    print('Posições repetidas na legenda:', posicoes_duplicadas)
    print('Nomes repetidos na legenda:', nomes_duplicados)

origens = df['ORIGEMREGISTRO'].value_counts(dropna=False)
resumo_origem = pd.DataFrame({'linhas': origens, 'percentual': (origens / len(df) * 100).round(1)})
display(resumo_origem)

,verificação,resultado
0,Colunas no CSV,66
1,Colunas numeradas na legenda,66
2,Nomes e posições conferem,True
3,Colunas extras no CSV,0
4,Colunas da legenda ausentes no CSV,0


,linhas,percentual
ORIGEMREGISTRO,,
ACADEMICA,44417,75.2
OCORRENCIA,14642,24.8


## Tipo semântico e grupo temático de cada coluna

Os tipos semânticos seguem a legenda. O grupo temático segue a organização da issue: identificação, tempo, curso/vínculo, desempenho, frequência, disciplinas, atividades complementares, cadastral e ocorrências. A v7 não contém grupo financeiro.

In [3]:
identificadores = [c for c in colunas_csv if c.startswith('HANDLE')]
datas = [c for c in colunas_csv if c.startswith('DATA')]
flags = [c for c in colunas_csv if c.startswith('TEVE')]
periodos_letivos = ['ANOSEMESTRE', 'INGRESSO']
categoricas = ['SIGLACURSO', 'TURNO', 'SEXO', 'MATRICULAEFETIVADA', 'CODIGOSTATUSALUNO', 'STATUSALUNO', 'ORIGEMREGISTRO']
continuas = ['CRENOSEMESTRE', 'CREATESEMESTRE', 'MEDIANAATESEMESTRE']
percentuais = ['IA04_01']
contagens = ['PERIODO', 'APROVEIT', 'QUANTIDADEOCORRENCIAS', 'QTDOCORRENCIASSEMDATA', 'IA04_02']
contagens += [c for c in colunas_csv if c.startswith('IA75_') or c in ['IA34_03', 'IA65_03'] or c.startswith('ATIVCOMPL_')]

tipo_por_coluna = {}
for grupo, colunas, tipo in [
    ('identificação', identificadores, 'identificador'),
    ('tempo', periodos_letivos, 'período letivo'),
    ('curso/vínculo', categoricas, 'categórica nominal'),
    ('desempenho', continuas, 'contínua'),
    ('desempenho', ['APROVEIT'], 'contagem'),
    ('curso/vínculo', ['PERIODO'], 'contagem ordinal'),
    ('frequência', percentuais, 'percentual'),
    ('frequência', ['IA04_02'], 'contagem'),
    ('disciplinas', [c for c in colunas_csv if c.startswith('IA75_') or c in ['IA34_03', 'IA65_03']], 'contagem'),
    ('atividades complementares', [c for c in colunas_csv if c.startswith('ATIVCOMPL_')], 'contagem'),
    ('ocorrências', flags, 'binária'),
    ('ocorrências', datas, 'data'),
    ('ocorrências', ['QUANTIDADEOCORRENCIAS', 'QTDOCORRENCIASSEMDATA'], 'contagem'),
]:
    for coluna in colunas:
        tipo_por_coluna[coluna] = (grupo, tipo)

# Sexo é o único dado cadastral; ORIGEMREGISTRO identifica a natureza da linha.
tipo_por_coluna['SEXO'] = ('cadastral', 'categórica nominal')
tipo_por_coluna['ORIGEMREGISTRO'] = ('identificação', 'categórica nominal')

def tipo_compativel(coluna, tipo):
    serie = df[coluna]
    valores = serie.dropna()
    if tipo == 'identificador':
        return is_integer_dtype(serie.dtype)
    if tipo == 'período letivo':
        return valores.astype(str).str.fullmatch(r'\d{4}/[12]').all()
    if tipo == 'data':
        return is_datetime64_any_dtype(serie.dtype)
    if tipo == 'binária':
        return is_numeric_dtype(serie.dtype) and set(valores.unique()).issubset({0, 1})
    if tipo in ['contagem', 'contagem ordinal']:
        return is_numeric_dtype(serie.dtype) and ((valores.astype(float) % 1) == 0).all()
    if tipo in ['contínua', 'percentual']:
        return is_numeric_dtype(serie.dtype)
    # CODIGOSTATUSALUNO é categórico embora seu código seja numérico.
    return True

assert set(tipo_por_coluna) == set(colunas_csv), (set(colunas_csv) - set(tipo_por_coluna), set(tipo_por_coluna) - set(colunas_csv))
grupos = {c: grupo for c, (grupo, _) in tipo_por_coluna.items()}
tabela_tipos = pd.DataFrame({
    'tipo_na_legenda': {c: t for c, (_, t) in tipo_por_coluna.items()},
    'dtype_inferido': df.dtypes.astype(str),
    'tipo_compativel': {c: tipo_compativel(c, t) for c, (_, t) in tipo_por_coluna.items()},
    'grupo_temático': grupos,
})
display(tabela_tipos)
tipos_incompativeis = [c for c in colunas_csv if not tabela_tipos.loc[c, 'tipo_compativel']]
print('Colunas com tipo incompatível:', tipos_incompativeis)

,tipo_na_legenda,dtype_inferido,tipo_compativel,grupo_temático
ANOSEMESTRE,período letivo,str,True,tempo
APROVEIT,contagem,float64,True,desempenho
ATIVCOMPL_APROVADAS,contagem,float64,True,atividades complementares
ATIVCOMPL_MATRICULADAS,contagem,float64,True,atividades complementares
ATIVCOMPL_REPROVADAS,contagem,float64,True,atividades complementares
...,...,...,...,...
TEVEREINGRESSO,binária,int64,True,ocorrências
TEVEREMATRICULA,binária,int64,True,ocorrências
TEVETRANCAMENTO,binária,int64,True,ocorrências
TEVETRANSAFASTAMENTO,binária,int64,True,ocorrências


Colunas com tipo incompatível: []


## Domínios, faixas e preenchimento

Para as colunas numéricas, o relatório compara mínimos e máximos às faixas descritas. Identificadores são excluídos dos cálculos de faixa. O preenchimento é calculado para as 66 colunas e comparado às porcentagens publicadas na legenda quando ela fornece esse valor.

In [4]:
faixas = {
    'PERIODO': (0, 10), 'CRENOSEMESTRE': (0, 100), 'CREATESEMESTRE': (0, 100),
    'MEDIANAATESEMESTRE': (0, 100), 'APROVEIT': (1, 23), 'QUANTIDADEOCORRENCIAS': (0, 2),
    'QTDOCORRENCIASSEMDATA': (0, 0), 'IA04_01': (0, 100), 'IA04_02': (0, 10),
    'IA75_01': (4, 69), 'IA75_02': (0, 69), 'IA75_03': (0, 12), 'IA75_04': (0, 13),
    'IA75_05': (0, 12), 'IA34_03': (0, 11), 'IA65_03': (0, 9),
    'ATIVCOMPL_MATRICULADAS': (0, 13), 'ATIVCOMPL_APROVADAS': (0, 12), 'ATIVCOMPL_REPROVADAS': (0, 10),
}
for coluna in flags:
    faixas[coluna] = (0, 1)

relatorio_faixas = []
for coluna, (min_esperado, max_esperado) in faixas.items():
    valores = df[coluna].dropna()
    minimo = valores.min() if not valores.empty else pd.NA
    maximo = valores.max() if not valores.empty else pd.NA
    compativel = valores.empty or (minimo >= min_esperado and maximo <= max_esperado)
    relatorio_faixas.append((coluna, min_esperado, max_esperado, minimo, maximo, compativel))
display(pd.DataFrame(relatorio_faixas, columns=['coluna', 'mínimo esperado', 'máximo esperado', 'mínimo observado', 'máximo observado', 'faixa confere']))

dominios = {
    'TURNO': {'Integral', 'Noturno'}, 'SEXO': {'M', 'F'},
    'MATRICULAEFETIVADA': {'S', 'N'},
    'STATUSALUNO': {'Ativo', 'Inativo', 'Matrícula Pendente', 'Em Mobilidade Acadêmica', 'Matrícula Alterada', 'Desconhecido'},
    'ORIGEMREGISTRO': {'ACADEMICA', 'OCORRENCIA'},
}
for coluna, esperado in dominios.items():
    observado = set(df[coluna].dropna().astype(str).unique())
    print(f'{coluna}: domínio confere={observado.issubset(esperado)}; categorias não documentadas={sorted(observado - esperado)}')
print(f'SIGLACURSO: 12 códigos conforme a legenda={df["SIGLACURSO"].nunique(dropna=True) == 12}; GAE observado={df["SIGLACURSO"].eq("GAE").any()}')
for coluna in periodos_letivos:
    partes = df[coluna].dropna().astype(str).str.extract(r'^(\d{4})/([12])$').astype(int)
    ordinal = partes[0] * 2 + partes[1] - 1
    print(f'{coluna}: valores entre 2009/1 e 2026/1={bool(ordinal.between(4018, 4052).all())}')

percentuais_legenda = {}
for linha in legenda.splitlines():
    partes = linha.split('|')
    if len(partes) < 4 or not partes[1].strip().isdigit():
        continue
    nome = partes[2].strip().strip('`')
    correspondencias = re.findall(r'(\d+(?:[.,]\d+)?)\s*%', '|'.join(partes[3:]))
    if nome in colunas_csv and correspondencias:
        percentuais_legenda[nome] = float(correspondencias[-1].replace(',', '.'))
# Seções de disciplinas e atividades complementares: preenchimento de 75,2% informado em prosa na legenda.
for coluna in colunas_csv:
    if coluna.startswith(('IA75_', 'ATIVCOMPL_')) or coluna in ['IA34_03', 'IA65_03']:
        percentuais_legenda[coluna] = 75.2

preenchimento_csv = (df.notna().mean() * 100).round(1)
relatorio_preenchimento = pd.DataFrame({
    'preenchimento_csv_%': preenchimento_csv,
    'preenchimento_legenda_%': pd.Series(percentuais_legenda),
})
relatorio_preenchimento['confere'] = [
    pd.NA if pd.isna(esperado) else round(float(preenchimento_csv[coluna]), 1) == round(float(esperado), 1)
    for coluna, esperado in relatorio_preenchimento['preenchimento_legenda_%'].items()
]
# As demais colunas não têm percentual individual publicado na legenda.
display(relatorio_preenchimento)
divergencias_preenchimento = [c for c in percentuais_legenda if round(float(preenchimento_csv[c]), 1) != round(percentuais_legenda[c], 1)]
print('Colunas com preenchimento divergente:', divergencias_preenchimento)
print('Colunas sem percentual de referência na legenda:', [c for c in colunas_csv if c not in percentuais_legenda])

,coluna,mínimo esperado,máximo esperado,mínimo observado,máximo observado,faixa confere
0,PERIODO,0,10,0.0,10.00,True
1,CRENOSEMESTRE,0,100,0.0,100.00,True
2,CREATESEMESTRE,0,100,0.0,100.00,True
3,MEDIANAATESEMESTRE,0,100,0.0,100.00,True
4,APROVEIT,1,23,1.0,23.00,True
5,QUANTIDADEOCORRENCIAS,0,2,0.0,2.00,True
6,QTDOCORRENCIASSEMDATA,0,0,0.0,0.00,True
7,IA04_01,0,100,0.0,100.56,False
8,IA04_02,0,10,0.0,10.00,True
9,IA75_01,4,69,4.0,69.00,True


TURNO: domínio confere=True; categorias não documentadas=[]
SEXO: domínio confere=True; categorias não documentadas=[]
MATRICULAEFETIVADA: domínio confere=True; categorias não documentadas=[]
STATUSALUNO: domínio confere=True; categorias não documentadas=[]
ORIGEMREGISTRO: domínio confere=True; categorias não documentadas=[]
SIGLACURSO: 12 códigos conforme a legenda=True; GAE observado=True


ANOSEMESTRE: valores entre 2009/1 e 2026/1=True
INGRESSO: valores entre 2009/1 e 2026/1=True


,preenchimento_csv_%,preenchimento_legenda_%,confere
ANOSEMESTRE,100.0,100.0,True
APROVEIT,14.9,14.9,True
ATIVCOMPL_APROVADAS,75.2,75.2,True
ATIVCOMPL_MATRICULADAS,75.2,75.2,True
ATIVCOMPL_REPROVADAS,75.2,75.2,True
...,...,...,...
TEVEREINGRESSO,100.0,NaN,<NA>
TEVEREMATRICULA,100.0,NaN,<NA>
TEVETRANCAMENTO,100.0,NaN,<NA>
TEVETRANSAFASTAMENTO,100.0,NaN,<NA>


Colunas com preenchimento divergente: []
Colunas sem percentual de referência na legenda: ['TEVEDESISTENCIA', 'TEVEABANDONO', 'TEVETRANCAMENTO', 'TEVECANCELAMENTO', 'TEVEDESLIGAMENTO', 'TEVETRANSAFASTAMENTO', 'TEVEDIPLOMACAO', 'TEVECONCLUSAOPROGRAMA', 'TEVEREINGRESSO', 'TEVEMUDANCACURSO', 'TEVEMIGRACAOCURRICULO', 'DATADESISTENCIA', 'DATAABANDONO', 'DATATRANCAMENTO', 'DATACANCELAMENTO', 'DATADESLIGAMENTO', 'DATATRANSAFASTAMENTO', 'DATADIPLOMACAO', 'DATACONCLUSAOPROGRAMA', 'DATAREINGRESSO', 'DATAMUDANCACURSO', 'DATAMIGRACAOCURRICULO', 'TEVEREMATRICULA', 'TEVEFALECIMENTO', 'TEVEPRAZOESGOTADO', 'TEVEAFASTAMENTOSEMOCORRENCIA', 'DATAREMATRICULA', 'DATAFALECIMENTO', 'DATAPRAZOESGOTADO', 'DATAAFASTAMENTOSEMOCORRENCIA']


## Categorias e concentração do PERIODO = 0

`value_counts()` é aplicado somente a colunas categóricas, nunca a identificadores. Categorias com menos de 10 registros têm seus nomes e contagens agrupados/suprimidos.

In [5]:
def value_counts_com_supressao(serie, minimo=10):
    contagens = serie.value_counts(dropna=False)
    visiveis = contagens[contagens >= minimo].astype(object).copy()
    pequenas = contagens[contagens < minimo]
    if not pequenas.empty:
        total_pequenas = int(pequenas.sum())
        visiveis.loc['categorias <10 agrupadas'] = total_pequenas if total_pequenas >= minimo else '<10 (total suprimido)'
    return visiveis

colunas_categoricas = periodos_letivos + categoricas
for coluna in colunas_categoricas:
    print(f'\n{coluna}')
    display(value_counts_com_supressao(df[coluna]).rename_axis('categoria').to_frame('frequência'))

siglas_curso = set(df['SIGLACURSO'].dropna().astype(str).unique())
periodo_zero = df.loc[df['PERIODO'].eq(0), 'SIGLACURSO'].dropna().astype(str)
contagem_zero = periodo_zero.value_counts()
n_zero = int(len(periodo_zero))
n_zero_gae = int(contagem_zero.get('GAE', 0))
print(f'GAE está entre as siglas observadas: {"GAE" in siglas_curso}')
print(f'Quantidade de siglas observadas: {len(siglas_curso)}')
if n_zero >= 10 and n_zero_gae >= 10:
    print(f'Participação de GAE entre as linhas com PERIODO = 0: {100 * n_zero_gae / n_zero:.1f}%')
else:
    print('Participação de GAE entre as linhas com PERIODO = 0: suprimida por haver grupo com menos de 10 registros.')
display(value_counts_com_supressao(periodo_zero).rename_axis('SIGLACURSO quando PERIODO = 0').to_frame('frequência'))


ANOSEMESTRE


,frequência
categoria,
2016/1,3449
2014/1,3202
2015/1,2476
2026/1,2453
2013/1,2140
2014/2,2136
2016/2,2121
2015/2,2078
2013/2,2047



INGRESSO


,frequência
categoria,
2014/1,3944
2012/1,3615
2013/1,3545
2015/1,3368
2011/1,3224
2010/1,2305
2018/1,2253
2019/1,2242
2016/1,2166



SIGLACURSO


,frequência
categoria,
GEC,13791
GET,12533
GEA,8801
GEB,6240
GES,5947
GEE,4369
TAI,3229
TGT,1811
GEP,1331


... (10 outputs left out)


ORIGEMREGISTRO


,frequência
categoria,
ACADEMICA,44417
OCORRENCIA,14642


GAE está entre as siglas observadas: True
Quantidade de siglas observadas: 12
Participação de GAE entre as linhas com PERIODO = 0: 72.7%


,frequência
SIGLACURSO quando PERIODO = 0,
GAE,24
categorias <10 agrupadas,<10 (total suprimido)


## Origem e chave de unicidade da linha

A legenda descreve a unidade como aluno + semestre + currículo. As duas combinações abaixo são testadas internamente; o notebook mostra somente quantas combinações se repetem, sem exibir identificadores.

In [6]:
chaves = {
    'aluno + semestre + HANDLECURRICULO': ['HANDLEALUNO', 'ANOSEMESTRE', 'HANDLECURRICULO'],
    'aluno + semestre + HANDLEALUNOCURRICULO': ['HANDLEALUNO', 'ANOSEMESTRE', 'HANDLEALUNOCURRICULO'],
}
resultados_chaves = []
for descricao, colunas_chave in chaves.items():
    tamanhos = df.groupby(colunas_chave, dropna=False, sort=False).size()
    duplicadas = int((tamanhos > 1).sum())
    resultados_chaves.append((descricao, duplicadas, duplicadas == 0))
display(pd.DataFrame(resultados_chaves, columns=['chave testada', 'combinações repetidas', 'chave única']))

print('Verificações concluídas. Consulte a tabela de preenchimento e as faixas para registrar divergências em docs/divergencias_legenda.md.')

,chave testada,combinações repetidas,chave única
0,aluno + semestre + HANDLECURRICULO,35,False
1,aluno + semestre + HANDLEALUNOCURRICULO,0,True


Verificações concluídas. Consulte a tabela de preenchimento e as faixas para registrar divergências em docs/divergencias_legenda.md.
